# Create and Upload I14Y Concept

## Codelist Entries

In [15]:
import json

# read JSON file to dict
with open('../data/output/parties.json', 'r') as file:
    parties = json.load(file)

In [17]:
import pandas as pd

# base vor codes json
codes = {
    "data": [],
}

for id, party in parties.items():

    if len(party.get("names")) > 0 and pd.isna(party.get("dissolution_date")):  # check if the party has names and is not deleted
        name = party.get("names")[-1]  # get the last name in the list
        # get the last version in the dictionary
        version = next(reversed(party["versions"].values()))  # get the last version in the list
        child_of = [relation for relation in version.get("relationships") if relation.get("type_label") == "child-of"]  # get all child_of relations
        code_entry = {
            "annotations": [],
            "code": str(id),  # get the id of the party
            "name": name["name"]
        }
        if len(child_of) > 0:
            child_of = child_of[0]  # get the first child_of relation
            code_entry["parentCode"] = str(child_of.get("party_to"))
        codes["data"].append(code_entry)

# write the dict to a JSON file
with open('../data/output/i14y_values.json', 'w') as file:
    json.dump(codes, file, indent=4)

## Upload

In [ ]:
from dotenv import load_dotenv

ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

### Get Token

In [ ]:
import requests
import os

def get_access_token(client_id,client_secret):
    data = {'grant_type': 'client_credentials'}
    response = requests.post("https://identity.i14y.a.c.bfs.admin.ch/realms/bfs-sis-a/protocol/openid-connect/token", data=data, verify=False, auth=(client_id, client_secret))
    return response.json()
 
access_token = 'Bearer ' + get_access_token(os.getenv('I14Y_CLIENT_ID'), os.getenv('I14Y_CLIENT_SECRET'))['access_token']

### Create Concept

In [ ]:
import requests

headers = {
    'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
    'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    'Authorization': access_token
}

file_path = "../data/output/i14y_concept.json" #state the right file name here
with open(file_path, 'r') as file:
    json_data = file.read()

url = 'https://api-a.i14y.admin.ch/api/partner/v1/concepts'

response = requests.post(url, headers=headers, data=json_data, verify=False)

if response.status_code == 201:
    print(f'Status-Code: {response.status_code}')
    concept_id = response.json()
    print(f'Concept Id: {concept_id}')
else:
    print(f"Error: {response.status_code} - {response.text}")

### Add Codelist Entries

In [ ]:
headers = { 
    'Authorization': access_token,
    'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
}

url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{concept_id}/codelist-entries/imports/json'

file_path = '../data/output/i14y_values.json' #state the right file name here
with open(file_path, 'rb') as file:
    files = {
        'file': (file_path, file, 'text/json')
    }
    response = requests.post(url, headers=headers, files=files, verify = False)

print(f'Status-Code: {response.status_code} - {response.text}')
print(response)
